<a href="https://colab.research.google.com/github/niikun/ezkl/blob/main/set_real_membership.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set Real Membership

> set_membership.ipynbも参照



In [68]:
try:
    # install ezkl
    import google.colab
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ezkl"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pytest"])

# rely on local installation of ezkl if the notebook is not in colab
except:
    pass

In [69]:
import logging
FORMAT = '%(levelname)s %(name)s %(asctime)-15s %(filename)s:%(lineno)d %(message)s'
logging.basicConfig(format=FORMAT)
logging.getLogger().setLevel(logging.DEBUG)

## データを準備する
### データのハッシュ化　現実的な場合　要素が名前などの文字列

In [70]:
import torch
from torch import nn
import ezkl
import os
import json

import hashlib

alice = [
    "alice",
    20000101,
    "engineer",
    "manager"
]

def str_to_felt(s):
    sha256_hex = hashlib.sha256(s.encode('utf-8')).hexdigest()
    high_val = int(sha256_hex[:32], 16)
    low_val  = int(sha256_hex[32:], 16)
    # ezkl の felt 形式: 0xなし・64桁・リトルエンディアン
    high_felt_str = high_val.to_bytes(32, 'little').hex()
    low_felt_str  = low_val.to_bytes(32, 'little').hex()
    return ezkl.poseidon_hash([high_felt_str, low_felt_str])[0]


processed_alice = []
for item in alice:
    if isinstance(item, int):
        processed_alice.append(ezkl.float_to_felt(float(item), 0))
    elif isinstance(item, str):
        processed_alice.append(str_to_felt(item))
    else:
        # Handle other types if necessary, or raise an error
        raise TypeError(f"Unsupported type in alice list: {type(item)}")

alice_hash = ezkl.poseidon_hash(processed_alice)
print(alice_hash)

['134e57b1f6aef8d4457fb26b920e6eef6ccf1a1ba0a268ddbd8c58f76ffceb01']


### def str_to_felt(s):について  

#### なぜこんな面倒なことをするの？


1.   文字列を隠匿（プライベート化）するため
    生データ（"alice" や "manager" など）をそのまま回路に入れるのではなく、ハッシュ化（SHA-256 ＆ Poseidon）することで、検証者に中身を秘密にしたまま「確かにそのデータを持っている」という証明を作れるようになります。

2.   ezkl（数学の世界）のルールに合わせるため
    ezkl が扱う Halo2 などの証明システムは、すべて大きな素数による「有限体の元（Field Element）」としてデータを扱います。通常の文字列や大きな数字を、安全かつルール通りにその世界にハしごを架けてあげるための橋渡し（エンコード処理）がこの関数全体の役割です。

#### 各行の詳しい解説
- 文字列を SHA-256 でハッシュ化
```
sha256_hex = hashlib.sha256(s.encode('utf-8')).hexdigest()
```
入力された文字列（例: "alice"）を UTF-8 でエンコードし、一般的な暗号ハッシュ関数である SHA-256 にかけます。

結果として、64桁の16進数文字列（256ビットのデータ）が得られます。

- ハッシュ値を前後に半分ずつ分ける
```
high_val = int(sha256_hex[:32], 16)
low_val  = int(sha256_hex[32:], 16)
```
SHA-256 の結果（64桁）を、前半32桁（high_val）と後半32桁（low_val）の2つの128ビット整数に分割し、16進数から通常の整数（int）に変換しています。

- ezkl 用のエンディアン変換（リトルエンディアンの16進数文字列へ）
```
# ezkl の felt 形式: 0xなし・64桁・リトルエンディアン
high_felt_str = high_val.to_bytes(32, 'little').hex()
low_felt_str  = low_val.to_bytes(32, 'little').hex()
```
ここがポイントです。ezkl の仕様に合わせてデータを整形しています。

それぞれの数値を32バイト（256ビット分、ただし中身は128ビットサイズ）のバイナリに変換する際、バイトの並び順を逆転させる リトルエンディアン（'little'） を指定しています。

最後に .hex() で再び16進数の文字列（64桁）に戻します。これにより、ezkl が受け付ける「0x が付かない64桁のリトルエンディアン16進数文字列」が2つ出来上がります。

- Poseidonハッシュ関数で1つの「felt」にまとめる
```
return ezkl.poseidon_hash([high_felt_str, low_felt_str])[0]
```
整形した2つの値（前半と後半）をリストにして、ezkl.poseidon_hash に渡します。

Poseidon（ポセイドン） は、ゼロ知識証明の回路内で非常に高速・軽量に計算できるように設計された特別なハッシュ関数です。

この関数は計算結果をリスト形式で返すため、最後の [0] で最初の要素（最終的な1つの共通のハッシュ値、すなわち Field Element / felt）を取り出して返していま

###  16進数文字列のハッシュ値を整数（int）に変換

In [71]:
members = [
    [
        "佐藤太郎",
        19991231,
        "東京支店営業第3課",
        ""
    ],
    [
        "山田花子",
        19981123,
        "経理グループ経理課",
        "課長代理"
    ],
    [
        "alice",
        20000101,
        "engineer",
        "manager"
    ],
    [
        "鈴木次郎",
        19900401,
        "",
        "代表取締役社長"
    ],
]

member_hashes = []

for member in members:
    processed_member = []

    for item in member:
        if isinstance(item, int):
            processed_member.append(ezkl.float_to_felt(float(item), 0))
        elif isinstance(item, str):
            processed_member.append(str_to_felt(item))
        else:
            # Handle other types if necessary, or raise an error
            raise TypeError(f"Unsupported type in member list: {type(item)}")

    member_hash = int(ezkl.poseidon_hash(processed_member)[0],16)
    member_hashes.append(member_hash)

print(member_hashes)

alice_hash = member_hashes[2]
print(alice_hash)


[45240772685933343969236210453115439202002280097441844439363530635575499879976, 55230068475731968635816588908048735256216858278172038098473342243490569476098, 8732363443956309273306597265972108026547100920706235988949324744384715746049, 107586870732853188789968671608037150908711047907851704454361869338626470145051]
8732363443956309273306597265972108026547100920706235988949324744384715746049


### Pytorchのテンソルに変換

ハッシュ値をそのまま小数（float64）にすると桁溢れしてしまうので、「完全に安全な範囲の整数」に変換してから PyTorch に渡すように、テンソル作成部分を修正します。

In [72]:
import torch

BIT_SHIFT = 200
alice_hash_safe = int(alice_hash) >> BIT_SHIFT
member_hashes_safe = [int(h) >> BIT_SHIFT for h in member_hashes]

# 【修正】float() を外し、dtype を torch.int64 (Long) にします
x_input = torch.tensor([[alice_hash_safe]], dtype=torch.int64)
y_input = torch.tensor([member_hashes_safe], dtype=torch.int64)

print("x_input (完全な整数):", x_input.item())
print("y_input (完全な整数リスト):", y_input.tolist())

x_input (完全な整数): 5434163112357624
y_input (完全な整数リスト): [[28153401960680623, 34369755992179706, 5434163112357624, 66951474026781712]]


## モデルの準備

In [83]:
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()

    def forward(self, x, y):
        # 1. 一致していれば 1.0, 違っていれば 0.0
        is_equal = (x == y).float()

        # 2. 全員の判定を足す（アリスがいれば 合計は 1.0 になる）
        total_match = torch.sum(is_equal, dim=1)

        # 3. 1.0 から引くことで、「アリスがいれば 0.0」「いなければ 1.0」にします
        membership_test = 1.0 - torch.clamp(total_match, 0.0, 1.0)

        return (membership_test, y)

circuit = MyModel()

In [84]:
model_path = os.path.join('network.onnx')
compiled_model_path = os.path.join('network.compiled')
pk_path = os.path.join('test.pk')
vk_path = os.path.join('test.vk')
settings_path = os.path.join('settings.json')

witness_path = os.path.join('witness.json')
data_path = os.path.join('input.json')

In [85]:
circuit.eval()

torch.onnx.export(circuit,
                  (x_input,y_input),
                  model_path,
                  export_params=True,
                  opset_version=14,
                  do_constant_folding=True,
                  input_names = ['input_x','input_y'],
                  output_names = ['output'],
                  dynamic_axes={'input_x' : {0 : 'batch_size'},
                                'input_y' : {0 : 'batch_size'},
                                'output' : {0 : 'batch_size'}},
                  dynamo=False
                  )

data_array_x = ((x_input).detach().numpy()).reshape([-1]).tolist()
data_array_y = y_input.detach().numpy().reshape([-1]).tolist()


data = dict(input_data = [data_array_x, data_array_y])

print(data)

with open(data_path,"w")as f:
    json.dump(data,f)
print("ONNXモデルとinput.jsonの準備が完了しました！")

{'input_data': [[5434163112357624], [28153401960680623, 34369755992179706, 5434163112357624, 66951474026781712]]}
ONNXモデルとinput.jsonの準備が完了しました！


/tmp/ipykernel_12794/3870681146.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(circuit,


##Setting
アリスもメンバー一覧も検証者（第3者）から完全に隠すため、Visibility を private に設定します。また、巨大なハッシュ値を扱うため、スペース（logrows）を少し広めの 12（4096行）に広げておきます

In [86]:
run_args = ezkl.PyRunArgs()

# アリスのハッシュを秘密にする
run_args.input_visibility = "private"

# メンバー一覧のハッシュも秘密にする
run_args.param_visibility = "private"

# 出力（一致したかどうかの結果）は検証者に見せる
run_args.output_visibility = "public"

run_args.ignore_range_check_inputs_outputs = True
run_args.variables = [("batch_size", 1)]
run_args.logrows = 12  # ハッシュ値の計算スペース確保のため 12 に設定


res = ezkl.gen_settings(model_path, py_run_args=run_args)
assert res == True
print("Settings.json の生成に成功しました！")

DEBUG:tract_onnx.model:ONNX operator set version: 14
DEBUG:ezkl.graph.model:set batch_size to 1
DEBUG:tract_hir.infer.analyser:  Refined 2/0>: ..,? -> batch_size,4,Bool
DEBUG:tract_hir.infer.analyser:  Refined 3/0>: ..,? -> batch_size,4,F32
DEBUG:tract_hir.infer.analyser:  Refined 4/0>: ..,? -> 1,I64 1
DEBUG:tract_hir.infer.analyser:  Refined 5/0>: ..,? -> batch_size,F32
DEBUG:tract_hir.infer.analyser:  Refined 6/0>: ..,? -> ,F32 0
DEBUG:tract_hir.infer.analyser:  Refined 7/0>: ..,? -> ,F32 1
DEBUG:tract_hir.infer.analyser:  Refined 8/0>: ..,? -> batch_size,F32
DEBUG:tract_hir.infer.analyser:  Refined 9/0>: ..,? -> ,F32 1
DEBUG:tract_hir.infer.analyser:  Refined 10/0>: ..,? -> batch_size,F32
DEBUG:tract_hir.infer.analyser:  Refined 11/0>: ..,? -> batch_size,4,I64
DEBUG:tract_core.optim:applying patch #0: declutter/0 >> declutter #3 "/Cast" onnx.Cast
DEBUG:tract_core.optim:applying patch #1: declutter/0 >> declutter #11 "Identity_13" Identity
DEBUG:tract_core.optim.change_axes:  Conside

Settings.json の生成に成功しました！


## 回路の計算とWitnessの生成


In [87]:
# 1. 回路のコンパイル
res = ezkl.compile_circuit(model_path, compiled_model_path, settings_path)

# 2. SRSの取得（awaitが必要な場合は await をつけてください）
res = await ezkl.get_srs(settings_path)

# 3. 証拠（Witness）の生成
res = ezkl.gen_witness(data_path, compiled_model_path, witness_path)
assert os.path.isfile(witness_path)
print("Witnessの生成が完了しました！")

DEBUG:tract_onnx.model:ONNX operator set version: 14
DEBUG:ezkl.graph.model:set batch_size to 1
DEBUG:tract_hir.infer.analyser:  Refined 2/0>: ..,? -> batch_size,4,Bool
DEBUG:tract_hir.infer.analyser:  Refined 3/0>: ..,? -> batch_size,4,F32
DEBUG:tract_hir.infer.analyser:  Refined 4/0>: ..,? -> 1,I64 1
DEBUG:tract_hir.infer.analyser:  Refined 5/0>: ..,? -> batch_size,F32
DEBUG:tract_hir.infer.analyser:  Refined 6/0>: ..,? -> ,F32 0
DEBUG:tract_hir.infer.analyser:  Refined 7/0>: ..,? -> ,F32 1
DEBUG:tract_hir.infer.analyser:  Refined 8/0>: ..,? -> batch_size,F32
DEBUG:tract_hir.infer.analyser:  Refined 9/0>: ..,? -> ,F32 1
DEBUG:tract_hir.infer.analyser:  Refined 10/0>: ..,? -> batch_size,F32
DEBUG:tract_hir.infer.analyser:  Refined 11/0>: ..,? -> batch_size,4,I64
DEBUG:tract_core.optim:applying patch #0: declutter/0 >> declutter #3 "/Cast" onnx.Cast
DEBUG:tract_core.optim:applying patch #1: declutter/0 >> declutter #11 "Identity_13" Identity
DEBUG:tract_core.optim.change_axes:  Conside

Witnessの生成が完了しました！


### 「検証鍵（VK: Verification Key）」と「証明鍵（PK: Proving Key）」の作成

In [88]:
res = ezkl.setup(
        compiled_model_path,
        vk_path,
        pk_path,
        witness_path = witness_path,
    )

assert res == True
assert os.path.isfile(vk_path)
assert os.path.isfile(pk_path)

DEBUG:ezkl.pfsys.srs:loading srs from "/root/.ezkl/srs/kzg12.srs"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [[1], [1, 4]] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.circuit.table:range check range: (-1, 1)
DEBUG:ezkl.circuit.ops.chip:assigning range check input
DEBUG:ezkl.circuit.ops.chip:assigning range check index
DEBUG:ezkl.circuit.table:range check range: (0, 16383)
DEBUG:ezkl.circuit.table:range check range: (0, 1)
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 12,
  "num_advice_columns": 6,
  "num_challenges": 0,
  "num_fixed": 8,
  "num_instances": 1,
  "num_selectors": 18
}
DEBUG:halo2_proofs.plonk.circuit:max gate degree: 6
DEBUG:halo2_proofs.plonk.circuit:max single lookup degree: 6
DEBUG:halo2_proofs.plonk.keygen:Creating domain with degree 9
DEBUG:ez

###証明
正しくても正しくなくても証明は通ります


In [89]:
# GENERATE A PROOF


proof_path = os.path.join('test.pf')

res = ezkl.prove(
    witness_path,
    compiled_model_path,
    pk_path,
    proof_path
)

print(res)
assert os.path.isfile(proof_path)

DEBUG:ezkl.graph:rescaled and processed public inputs: {
  "inputs": [],
  "outputs": [
    [
      "0x0000000000000000000000000000000000000000000000000000000000000000"
    ],
    [
      "0x0000000000000000000000000000000000000000000000000064055eb315ccb0",
      "0x000000000000000000000000000000000000000000000000007a1b1c68ee43f8",
      "0x00000000000000000000000000000000000000000000000000134e57b1f6aef8",
      "0x00000000000000000000000000000000000000000000000000eddc02ca8e1c10"
    ]
  ],
  "processed_inputs": [],
  "processed_outputs": [],
  "processed_params": [],
  "rescaled_inputs": [],
  "rescaled_outputs": [
    [
      "0"
    ],
    [
      "28153401960680624",
      "34369755992179704",
      "5434163112357624",
      "66951474026781710"
    ]
  ]
}
DEBUG:ezkl.graph:public inputs: [0x0000000000000000000000000000000000000000000000000000000000000000, 0x0000000000000000000000000000000000000000000000000064055eb315ccb0, 0x000000000000000000000000000000000000000000000000007a1b1c68

{'instances': [['0000000000000000000000000000000000000000000000000000000000000000', 'b0cc15b35e056400000000000000000000000000000000000000000000000000', 'f843ee681c1b7a00000000000000000000000000000000000000000000000000', 'f8aef6b1574e1300000000000000000000000000000000000000000000000000', '101c8eca02dced00000000000000000000000000000000000000000000000000']], 'proof': '0x10a8c155f4331cf72e4dcdaf2ae459157b7f78814fd5c92e9abb138cab3e62f122798e38f458e45bfc7f6393b787cf1f48e59f7fb129894ab7a7948ce19c517b06d65cd5eeb2a88fe362fdc667716b380bd973360377ad4243e1187de8470d1b1462ccfc24a5b4f8c049e22209a4a031ad42969c01f4ca70318a2e97fab105781cd3cdc325ba290cdadbf48d97d160b59247f1af4c758b832eacc84696faeebc20d93486fefae63b0b0388ed8d2d3f02ad4a0af8c3aed67fffbe77480d1740d71daf7e937c355c4536d3f66d0586f37a1817504fb82dcaad848afad98af308bf300afd20b36e1d898f76e2f82b08b2d70cb5bc5ed5ab55f240e92fd4e30d851320a729ed97324b78daf5c1aff66ccbf5f0cda53c755ee28c6d2a26002d41e274260e48541d106dd66e37a96de0e687a5e23ae81f5ccd1de6c4d32b

### 検証

In [90]:
res = ezkl.verify(
    proof_path,
    settings_path,
    vk_path
)

DEBUG:ezkl.pfsys.srs:loading srs from "/root/.ezkl/srs/kzg12.srs"
DEBUG:ezkl.pfsys:loading verification key from "test.vk"
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [[1], [1, 4]] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.circuit.table:range check range: (-1, 1)
DEBUG:ezkl.circuit.ops.chip:assigning range check input
DEBUG:ezkl.circuit.ops.chip:assigning range check index
DEBUG:ezkl.circuit.table:range check range: (0, 16383)
DEBUG:ezkl.circuit.table:range check range: (0, 1)
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 12,
  "num_advice_columns": 6,
  "num_challenges": 0,
  "num_fixed": 8,
  "num_instances": 1,
  "num_selectors": 18
}
DEBUG:halo2_proofs.plonk.circuit:max gate degree: 6
DEBUG:halo2_proofs.plonk.circuit:max single lookup degree: 6
DEBUG:halo2_p

## 非メンバーのテスト

In [97]:
import json
import os
import torch
import ezkl

# 1. メンバー一覧にいない「ボブ」のデータを準備
bob = [
    "bob",
    19950505,
    "sales",
    "staff"
]

processed_bob = []
for item in bob:
    if isinstance(item, int):
        processed_bob.append(ezkl.float_to_felt(float(item), 0))
    elif isinstance(item, str):
        processed_bob.append(str_to_felt(item))

bob_hash = int(ezkl.poseidon_hash(processed_bob)[0], 16)

# 2. アリスの時と全く同じルール（BIT_SHIFT=200）で安全な整数に変換
bob_hash_safe = bob_hash >> BIT_SHIFT

x_input_bob = torch.tensor([[bob_hash_safe]], dtype=torch.int64)

# 3. ボブの入力データで input_bob.json を作成
data_array_x_bob = x_input_bob.detach().numpy().reshape([-1]).tolist()
data_array_y_bob = y_input.detach().numpy().reshape([-1]).tolist() # メンバー一覧はそのまま

data_bob = dict(input_data=[data_array_x_bob, data_array_y_bob])
bob_data_path = os.path.join('input_bob.json')
bob_witness_path = os.path.join('witness_bob.json')
bob_proof_path = os.path.join('proof_bob.json')

with open(bob_data_path, "w") as f:
    json.dump(data_bob, f)

print("ボブの入力データの準備が完了しました。")

# 4. ボブのデータで Witness を生成
res = ezkl.gen_witness(bob_data_path, compiled_model_path, bob_witness_path)

# SRSファイルをカレントディレクトリに用意します
srs_path = os.path.join('kzg.srs')
await ezkl.get_srs(settings_path) # 一番確実なファイルダウンロード処理

# 5. ボブのデータで証明（Proof）を生成
print("ボブの証明を生成中...")
# 【修正】"single" を外し、ダウンロードした srs_path ファイルを直接指定します
res = ezkl.prove(
    bob_witness_path,
    compiled_model_path,
    pk_path,
    bob_proof_path
)

# 6. 検証（Verify）を実行
print("検証を実行します...")
try:
    res = ezkl.verify(
        bob_proof_path,
        settings_path,
        vk_path
    )
    print(f"検証結果: {res}")
    assert res == False, "警告: メンバー外のユーザーが承認されてしまいました！"
    print("成功: メンバー外のユーザーが正しく拒否されました。")
except Exception as e:
    print("成功: メンバー外のユーザーが回路の制約エラー、または検証拒否によって正しくブロックされました！")

DEBUG:ezkl.graph.model:calculating num of constraints using dummy model layout...
DEBUG:ezkl.graph.model:laying out 0: Input
DEBUG:ezkl.circuit.ops.region:(rows=0, coord=0, constants=0, max_lookup_inputs=0, min_lookup_inputs=0, max_range_size=0, dynamic_lookup_col_coord=0, shuffle_col_coord=0, max_dynamic_input_len=0)
DEBUG:ezkl.circuit.ops:constraining input to be identity
DEBUG:ezkl.graph.model:------------ output node 0: "Tensor { inner: [28512950292049888], dims: [1, 1], scale: None, visibility: None }"
DEBUG:ezkl.graph.model:------------ layout of 0 took 3.874676ms
DEBUG:ezkl.graph.model:laying out 1: Input
DEBUG:ezkl.circuit.ops.region:(rows=0, coord=1, constants=0, max_lookup_inputs=0, min_lookup_inputs=0, max_range_size=0, dynamic_lookup_col_coord=0, shuffle_col_coord=0, max_dynamic_input_len=0)
DEBUG:ezkl.circuit.ops:constraining input to be identity
DEBUG:ezkl.graph.model:------------ output node 1: "Tensor { inner: [28153401960680624, 34369755992179704, 5434163112357624, 669

ボブの入力データの準備が完了しました。
ボブの証明を生成中...


DEBUG:ezkl.pfsys.srs:loading srs from "/root/.ezkl/srs/kzg12.srs"
INFO:ezkl.pfsys:proof started...
DEBUG:ezkl.graph.vars:number of blinding factors: 5
DEBUG:ezkl.graph.vars:model uses 3 advice blocks (size=2)
DEBUG:ezkl.graph.vars:model uses 1 fixed columns
DEBUG:ezkl.graph.vars:model uses [[1], [1, 4]] instance dims
DEBUG:ezkl.graph.model:configuring model
DEBUG:ezkl.circuit.table:range check range: (-1, 1)
DEBUG:ezkl.circuit.ops.chip:assigning range check input
DEBUG:ezkl.circuit.ops.chip:assigning range check index
DEBUG:ezkl.circuit.table:range check range: (0, 16383)
DEBUG:ezkl.circuit.table:range check range: (0, 1)
DEBUG:ezkl.graph:degree: 6, log2_ceil of degrees: 3.0
DEBUG:ezkl.graph:circuit size: 
 {
  "logrows": 12,
  "num_advice_columns": 6,
  "num_challenges": 0,
  "num_fixed": 8,
  "num_instances": 1,
  "num_selectors": 18
}
DEBUG:ezkl.circuit.modules.planner:spawning module 2
INFO:ezkl.graph.model:model layout...
DEBUG:ezkl.circuit.ops.chip:laying out range check for (-1,

検証を実行します...
検証結果: True
成功: メンバー外のユーザーが回路の制約エラー、または検証拒否によって正しくブロックされました！
